# Relationship Validation

### Loading Data

In [ ]:
%pip install -q kagglehub

from pathlib import Path
import pandas as pd
import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("olistbr/brazilian-ecommerce")
)

orders_file = next(dataset_path.rglob("olist_orders_dataset.csv"))
RAW_DIR = orders_file.parent

csv_files = sorted(RAW_DIR.glob("*.csv"))

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


### Loading Tables

In [ ]:
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")

### Check foreign-key relationships

In [ ]:
def check_fk(child_df, child_col, parent_df, parent_col, relationship_name):
    unmatched = child_df.loc[
        ~child_df[child_col].isin(parent_df[parent_col]),
        child_col
    ]

    print(relationship_name)
    print(f"Child rows: {len(child_df):,}")
    print(f"Unmatched rows: {len(unmatched):,}")
    print(f"Unique unmatched values: {unmatched.nunique():,}")
    print("-" * 50)


check_fk(
    items, "order_id",
    orders, "order_id",
    "Order Items -> Orders"
)

check_fk(
    reviews, "order_id",
    orders, "order_id",
    "Reviews -> Orders"
)

check_fk(
    payments, "order_id",
    orders, "order_id",
    "Payments -> Orders"
)

check_fk(
    items, "seller_id",
    sellers, "seller_id",
    "Order Items -> Sellers"
)

check_fk(
    items, "product_id",
    products, "product_id",
    "Order Items -> Products"
)

check_fk(
    orders, "customer_id",
    customers, "customer_id",
    "Orders -> Customers"
)

Order Items -> Orders
Child rows: 112,650
Unmatched rows: 0
Unique unmatched values: 0
--------------------------------------------------
Reviews -> Orders
Child rows: 99,224
Unmatched rows: 0
Unique unmatched values: 0
--------------------------------------------------
Payments -> Orders
Child rows: 103,886
Unmatched rows: 0
Unique unmatched values: 0
--------------------------------------------------
Order Items -> Sellers
Child rows: 112,650
Unmatched rows: 0
Unique unmatched values: 0
--------------------------------------------------
Order Items -> Products
Child rows: 112,650
Unmatched rows: 0
Unique unmatched values: 0
--------------------------------------------------
Orders -> Customers
Child rows: 99,441
Unmatched rows: 0
Unique unmatched values: 0
--------------------------------------------------


### Check one-to-many relationships

In [ ]:
relationship_counts = {
    "Items per order": items.groupby("order_id").size(),
    "Reviews per order": reviews.groupby("order_id").size(),
    "Payments per order": payments.groupby("order_id").size(),
    "Items per seller": items.groupby("seller_id").size(),
    "Items per product": items.groupby("product_id").size(),
    "Orders per customer_id": orders.groupby("customer_id").size()
}

for name, counts in relationship_counts.items():
    print(name)
    print(f"Minimum: {counts.min()}")
    print(f"Maximum: {counts.max()}")
    print(f"IDs with more than 1 related row: {(counts > 1).sum():,}")
    print("-" * 50)

Items per order
Minimum: 1
Maximum: 21
IDs with more than 1 related row: 9,803
--------------------------------------------------
Reviews per order
Minimum: 1
Maximum: 3
IDs with more than 1 related row: 547
--------------------------------------------------
Payments per order
Minimum: 1
Maximum: 29
IDs with more than 1 related row: 2,961
--------------------------------------------------
Items per seller
Minimum: 1
Maximum: 2033
IDs with more than 1 related row: 2,586
--------------------------------------------------
Items per product
Minimum: 1
Maximum: 527
IDs with more than 1 related row: 14,834
--------------------------------------------------
Orders per customer_id
Minimum: 1
Maximum: 1
IDs with more than 1 related row: 0
--------------------------------------------------


### Check orders missing related records

In [ ]:
print(
    "Orders without items:",
    (~orders["order_id"].isin(items["order_id"])).sum()
)

print(
    "Orders without reviews:",
    (~orders["order_id"].isin(reviews["order_id"])).sum()
)

print(
    "Orders without payments:",
    (~orders["order_id"].isin(payments["order_id"])).sum()
)

Orders without items: 775
Orders without reviews: 768
Orders without payments: 1


### Customer relationship check

In [ ]:
print(
    "Duplicate customer_id values in customers:",
    customers["customer_id"].duplicated().sum()
)

print(
    "Duplicate customer_id values in orders:",
    orders["customer_id"].duplicated().sum()
)

print(
    "Orders with customer_id not found in customers:",
    (~orders["customer_id"].isin(customers["customer_id"])).sum()
)

Duplicate customer_id values in customers: 0
Duplicate customer_id values in orders: 0
Orders with customer_id not found in customers: 0


# Notes

- All tested foreign-key relationships matched successfully; no unmatched IDs were found.
- Orders can contain multiple items, with up to 21 items in one order.
- Some orders have multiple reviews (up to 3) and multiple payment records (up to 29), which will create join risks later.
- Seller and product relationships are strongly one-to-many, with sellers linked to many items and products appearing in multiple order-item records.
- `customer_id` is one-to-one with orders in this dataset; no duplicate or unmatched customer IDs were found.
- 775 orders have no item records, 768 have no review records, and 1 order has no payment record.
- No data were modified during validation.

---

# Join-Risk Analysis

### Count Rows

In [ ]:
print("Orders:", len(orders))
print("Items:", len(items))
print("Reviews:", len(reviews))
print("Payments:", len(payments))

Orders: 99441
Items: 112650
Reviews: 99224
Payments: 103886


### Comparing what happens with simple joins

In [ ]:
orders_items = orders.merge(
    items,
    on="order_id",
    how="left"
)

print("Orders -> Items rows:", len(orders_items))

Orders -> Items rows: 113425


In [ ]:
orders_items_reviews = orders_items.merge(
    reviews[["order_id", "review_id", "review_score"]],
    on="order_id",
    how="left"
)

print("Orders -> Items -> Reviews rows:", len(orders_items_reviews))

Orders -> Items -> Reviews rows: 114092


In [ ]:
orders_items_reviews_payments = orders_items_reviews.merge(
    payments[["order_id", "payment_sequential", "payment_value"]],
    on="order_id",
    how="left"
)

print(
    "Orders -> Items -> Reviews -> Payments rows:",
    len(orders_items_reviews_payments)
)

Orders -> Items -> Reviews -> Payments rows: 119143


### Checking how much multiplication happens per order

In [ ]:
join_counts = (
    orders_items_reviews_payments
    .groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

print("Maximum joined rows for one order:", join_counts.max())
print("Orders producing more than 1 joined row:", (join_counts > 1).sum())

display(join_counts.head(10))

Maximum joined rows for one order: 63
Orders producing more than 1 joined row: 12947


,0
order_id,
895ab968e7bb0d5659d16cd74cd1650c,63
fedcd9f7ccdc8cba3a18defedd1a5547,38
fa65dad1b0e818e3ccc5cb0e39231352,29
ccf804e764ed5650cd8759557269dc13,26
6d58638e32674bebee793a47ac4cbadc,24
465c2e1bee4561cb39e0db8c5993aafc,24
c6492b842ac190db807c15aff21a7dd6,24
a3725dfe487d359b5be08cac48b64ec5,24
68986e4324f6a21481df4e6e89abcf01,24


###  Testing a safer approach

In [ ]:
payment_summary = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_record_count=("payment_sequential", "count")
    )
)

review_summary = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_record_count=("review_id", "count")
    )
)

print("Payment summary rows:", len(payment_summary))
print("Review summary rows:", len(review_summary))

Payment summary rows: 99440
Review summary rows: 98673


### Joining those summaries to orders

In [ ]:
safe_order_level = (
    orders
    .merge(payment_summary, on="order_id", how="left")
    .merge(review_summary, on="order_id", how="left")
)

print("Safe order-level rows:", len(safe_order_level))
print("Original order rows:", len(orders))

Safe order-level rows: 99441
Original order rows: 99441


# Notes

- Directly joining orders to items, reviews, and payments increases row counts because these tables contain one-to-many relationships.
- The full direct join produced 119,143 rows from 99,441 original orders.
- 12,947 orders produced multiple joined rows, with one order expanding to 63 rows.
- Reviews and payments can be safely summarized to one row per order before joining.
- After aggregating reviews and payments first, the order-level dataset remained at 99,441 rows.
- Future joins will use aggregation where needed to avoid inflated counts, sales, payments, or review measures.

---

# Geolocation and Review Decisions

In [ ]:
geo = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")

geo_summary = (
    geo
    .drop_duplicates()
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean")
    )
)

print("Raw geolocation rows:", len(geo))
print("Reduced geolocation rows:", len(geo_summary))
print(
    "Unique ZIP prefixes:",
    geo_summary["geolocation_zip_code_prefix"].nunique()
)

Raw geolocation rows: 1000163
Reduced geolocation rows: 19015
Unique ZIP prefixes: 19015


### Checking seller counts per order

In [ ]:
seller_counts_per_order = (
    items
    .groupby("order_id")["seller_id"]
    .nunique()
)

print(
    "Single-seller orders:",
    (seller_counts_per_order == 1).sum()
)

print(
    "Multi-seller orders:",
    (seller_counts_per_order > 1).sum()
)

print(
    "Maximum sellers in one order:",
    seller_counts_per_order.max()
)

Single-seller orders: 97388
Multi-seller orders: 1278
Maximum sellers in one order: 5


In [ ]:
single_seller_pct = (
    (seller_counts_per_order == 1).mean() * 100
)

print(
    f"Percent of item-containing orders that are single-seller: "
    f"{single_seller_pct:.2f}%"
)

Percent of item-containing orders that are single-seller: 98.70%


In [ ]:
single_seller_orders = seller_counts_per_order[
    seller_counts_per_order == 1
].index

print(
    "Orders eligible for seller-review analysis:",
    len(single_seller_orders)
)

Orders eligible for seller-review analysis: 97388


# Notes

- Geolocation was reduced from 1,000,163 rows to 19,015 unique ZIP prefixes by averaging coordinates within each ZIP prefix.
- This creates a one-row-per-ZIP lookup that can be joined safely without multiplying records.
- 97,388 orders are single-seller and 1,278 are multi-seller.
- 98.70% of item-containing orders are single-seller.
- Seller-level review analysis will use single-seller orders so an order-level review is not incorrectly attributed to multiple sellers.
- No raw data were modified.

---

# Discover Stage Summary

### Main Data Sources
- Orders, order items, reviews, sellers, products, and customers are the core tables.
- Geolocation and category translation are supporting tables.
- Payments may be used if they add useful information to the analysis.

### Key Data Structure Decisions
- Order-level, item-level, and seller-level analyses will be kept separate when appropriate.
- Reviews and payments should be aggregated before joining to order-level data.
- Raw geolocation should not be joined directly because ZIP prefixes repeat many times.
- Geolocation will be reduced to one representative latitude/longitude per ZIP prefix.
- Seller-level review analysis will use single-seller orders only.

### Main Data Limitations Identified
- Some review IDs are associated with multiple orders.
- Geolocation contains substantial duplication.
- Review comment fields contain substantial missing data.
- Some order date fields and product fields contain missing values.
- Some orders do not have corresponding item, review, or payment records.
- Multi-seller orders create ambiguity when assigning an order-level review to a specific seller.

### Planned Analysis Variables
Customer experience:
- review_score
- delivery_days
- delivery_delay_days
- late_delivery_flag
- freight_percentage
- order_value

Seller performance:
- seller_order_count
- seller_items_sold
- seller_total_sales
- seller_average_review
- seller_late_delivery_rate
- seller_category_count

### Next Stage
Begin data preparation: clean fields, convert dates, investigate missing values,
create derived variables, and build analysis-ready datasets.